# 10-01 Agent 记忆系统

Agent 的记忆能力决定了它是否能进行连贯的多轮对话、积累经验。本节实现三种核心记忆机制。

**本节目标**：
- 理解短期记忆（对话缓冲）、长期记忆（向量存储）、情景记忆的区别
- 实现 ConversationBufferMemory 和 ConversationSummaryMemory
- 实现基于余弦相似度的长期记忆检索

---

In [ ]:
import sys, os, math
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("../.env")

from utils.llm_client import call_llm

# ========== 1. 短期记忆：对话缓冲 ==========

class ConversationBufferMemory:
    """最简单的记忆：保存完整对话历史，窗口截断防止 token 爆炸。"""

    def __init__(self, max_turns: int = 10):
        self.history: list[dict] = []
        self.max_turns = max_turns

    def add(self, role: str, content: str):
        self.history.append({"role": role, "content": content})
        # 窗口截断：只保留最近 max_turns 轮
        if len(self.history) > self.max_turns * 2:
            self.history = self.history[-self.max_turns * 2:]

    def get_context(self) -> str:
        return "\n".join(f"[{m['role']}]: {m['content']}" for m in self.history)

    def clear(self):
        self.history.clear()


class ConversationSummaryMemory:
    """摘要记忆：当对话过长时，用 LLM 压缩历史为摘要，节省 token。"""

    def __init__(self, max_turns_before_summary: int = 6):
        self.history: list[dict] = []
        self.summary: str = ""
        self.max_turns = max_turns_before_summary

    def add(self, role: str, content: str):
        self.history.append({"role": role, "content": content})
        if len(self.history) > self.max_turns * 2:
            self._compress()

    def _compress(self):
        """将前半部分对话压缩为摘要（实际项目中调用 LLM）。"""
        old_msgs = self.history[:len(self.history) // 2]
        old_text = "\n".join(f"{m['role']}: {m['content']}" for m in old_msgs)
        # 模拟 LLM 摘要（实际用 call_llm）
        self.summary += f"\n[历史摘要] 用户讨论了以下内容: {old_text[:100]}..."
        self.history = self.history[len(self.history) // 2:]
        print(f"  [压缩] 对话已压缩，当前保留 {len(self.history)} 条消息")

    def get_context(self) -> str:
        parts = []
        if self.summary:
            parts.append(f"历史摘要: {self.summary}")
        parts.append("近期对话:")
        parts.extend(f"[{m['role']}]: {m['content']}" for m in self.history)
        return "\n".join(parts)


# ---- 演示 ----
buffer = ConversationBufferMemory(max_turns=3)
buffer.add("user", "帮我写一个B站游戏联运广告")
buffer.add("assistant", "好的，目标受众是什么年龄段？")
buffer.add("user", "18-25岁，男性为主")
buffer.add("assistant", "推荐文案：'开黑不孤单，B站联运新游首发'")
print("=== Buffer Memory ===")
print(buffer.get_context())

print("\n=== Summary Memory ===")
summary_mem = ConversationSummaryMemory(max_turns_before_summary=2)
for i, (role, msg) in enumerate([
    ("user", "B站广告投放预算多少？"), ("assistant", "建议日预算500-2000元"),
    ("user", "目标人群呢？"), ("assistant", "建议定向18-30岁ACG用户"),
    ("user", "效果怎么追踪？"), ("assistant", "可通过B站广告后台看CTR/CVR")
]):
    summary_mem.add(role, msg)
print(summary_mem.get_context())

In [ ]:
# ========== 2. 长期记忆：向量存储 + 余弦相似度检索 ==========

def simple_embedding(text: str, dim: int = 64) -> list[float]:
    """极简文本嵌入（教学用）：基于字符哈希生成固定维度向量。
    实际项目中应使用 OpenAI text-embedding-3-small 等专业模型。"""
    vec = [0.0] * dim
    for i, ch in enumerate(text):
        vec[hash(ch) % dim] += 1.0 / (i + 1)
    # L2 归一化
    norm = math.sqrt(sum(x * x for x in vec)) or 1.0
    return [x / norm for x in vec]


def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0


class VectorLongTermMemory:
    """长期记忆：将经验存入向量库，按语义检索最相关的记忆。"""

    def __init__(self):
        self.memories: list[dict] = []  # {"text": ..., "embedding": ..., "metadata": ...}

    def store(self, text: str, metadata: dict | None = None):
        emb = simple_embedding(text)
        self.memories.append({"text": text, "embedding": emb, "metadata": metadata or {}})

    def search(self, query: str, top_k: int = 3) -> list[dict]:
        q_emb = simple_embedding(query)
        scored = [(m, cosine_similarity(q_emb, m["embedding"])) for m in self.memories]
        scored.sort(key=lambda x: x[1], reverse=True)
        return [{"text": m["text"], "score": round(s, 3), **m["metadata"]} for m, s in scored[:top_k]]


# ---- 演示：B站广告经验库 ----
ltm = VectorLongTermMemory()

experiences = [
    ("游戏广告用二次元风格素材CTR提升40%", {"category": "游戏", "metric": "CTR"}),
    ("教育类广告在晚8点投放转化率最高", {"category": "教育", "metric": "CVR"}),
    ("美妆广告搭配UP主测评视频效果最好", {"category": "美妆", "metric": "ROI"}),
    ("3C产品广告标题含价格信息点击率更高", {"category": "3C", "metric": "CTR"}),
    ("食品广告在节假日投放ROI翻倍", {"category": "食品", "metric": "ROI"}),
]

for text, meta in experiences:
    ltm.store(text, meta)

# 检索
query = "如何提升游戏广告的点击率"
results = ltm.search(query, top_k=3)
print(f"查询: '{query}'")
print("检索结果:")
for r in results:
    print(f"  [{r['score']:.3f}] {r['text']}")

In [ ]:
# ========== 3. 情景记忆：记录成功/失败的完整交互 ==========

class EpisodicMemory:
    """情景记忆：记录完整的任务交互片段，包括结果和反馈。
    Agent 可以回顾过去的成功/失败经验来指导当前决策。"""

    def __init__(self):
        self.episodes: list[dict] = []

    def record(self, task: str, actions: list[str], outcome: str, success: bool):
        self.episodes.append({
            "task": task, "actions": actions,
            "outcome": outcome, "success": success
        })

    def recall_similar(self, task: str, only_success: bool = True) -> list[dict]:
        """根据任务描述召回相似的历史情景。"""
        filtered = [e for e in self.episodes if not only_success or e["success"]]
        # 简化匹配：关键词重合度
        task_words = set(task)
        scored = []
        for ep in filtered:
            overlap = len(task_words & set(ep["task"])) / max(len(task_words), 1)
            scored.append((ep, overlap))
        scored.sort(key=lambda x: x[1], reverse=True)
        return [ep for ep, _ in scored[:3]]


# ---- 演示 ----
em = EpisodicMemory()
em.record(
    task="为B站游戏联运写广告文案",
    actions=["分析目标用户", "生成3版文案", "A/B测试选优"],
    outcome="最终CTR 3.2%，高于行业均值", success=True
)
em.record(
    task="为B站教育课程写广告",
    actions=["直接生成文案", "未做用户分析"],
    outcome="CTR仅0.8%，低于预期", success=False
)
em.record(
    task="为B站游戏直播写推广文案",
    actions=["参考游戏联运成功案例", "加入直播元素", "UP主定制"],
    outcome="CTR 4.1%，效果优秀", success=True
)

print("=== 召回成功经验 ===")
similar = em.recall_similar("为B站新游戏写广告", only_success=True)
for ep in similar:
    print(f"  任务: {ep['task']}")
    print(f"  步骤: {' → '.join(ep['actions'])}")
    print(f"  结果: {ep['outcome']}\n")

## 记忆架构对比

| 记忆类型 | 存储内容 | 实现方式 | 容量 | 典型场景 |
|----------|----------|----------|------|----------|
| **短期记忆 (Buffer)** | 最近N轮对话原文 | 列表 + 窗口截断 | 小（受 token 限制） | 多轮对话上下文 |
| **短期记忆 (Summary)** | 历史摘要 + 近期原文 | LLM 压缩 | 中 | 长对话场景 |
| **长期记忆 (Vector)** | 经验/知识片段 | 向量数据库 + 语义检索 | 大（无上限） | 知识积累、经验复用 |
| **情景记忆 (Episodic)** | 完整任务交互记录 | 结构化存储 + 相似匹配 | 中 | 从历史成败中学习 |

## 面试速记

| 问题 | 要点 |
|------|------|
| Buffer vs Summary Memory | Buffer 保留原文精确但耗 token；Summary 节省 token 但有信息损失 |
| 长期记忆如何实现 | 向量数据库（Chroma/Pinecone）+ Embedding 模型，语义检索 top-k |
| 情景记忆的价值 | 让 Agent 从过去的成功/失败中学习，类似人类的经验积累 |
| 记忆系统设计原则 | 分层（短期+长期）、按需检索（不要全塞进 prompt）、定期清理过期记忆 |

**下一节**: `02_agent_evaluation.ipynb` — Agent 评估体系